# Price Coverage Audit

This notebook validates daily price coverage for every symbol in `silver/dim_company` against an expected start date rule:

`expected_start_date = max(coalesce(ipo_date, floor_date), floor_date)`

Default floor date is `1999-01-01`.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

BUCKET = os.environ['R2_BUCKET_NAME']
ACCT = os.environ['R2_ACCOUNT_ID']
AK = os.environ['R2_ACCESS_KEY_ID']
SK = os.environ['R2_SECRET_ACCESS_KEY']

def open_con() -> duckdb.DuckDBPyConnection:
    c = duckdb.connect()
    c.execute('INSTALL httpfs; LOAD httpfs;')
    c.execute(f"SET s3_endpoint = '{ACCT}.r2.cloudflarestorage.com';")
    c.execute(f"SET s3_access_key_id = '{AK}';")
    c.execute(f"SET s3_secret_access_key = '{SK}';")
    c.execute("SET s3_region = 'auto';")
    c.execute("SET s3_url_style = 'path';")
    return c

def silver(path: str) -> str:
    return f"read_parquet('s3://{BUCKET}/silver/{path}', union_by_name=true)"

con = open_con()
print('Connected to R2 via DuckDB')

Connected to R2 via DuckDB


In [13]:
# Parameters
FLOOR_DATE = '2000-01-01'
SYMBOL_FILTER = None  # Example: ['AAPL', 'USLM', 'JPM']
TOP_N = 50

if SYMBOL_FILTER:
    symbols_sql = '(' + ', '.join([f"'{s}'" for s in SYMBOL_FILTER]) + ')'
    symbol_where = f'WHERE d.symbol IN {symbols_sql}'
else:
    symbol_where = ''

print('FLOOR_DATE =', FLOOR_DATE)
print('SYMBOL_FILTER =', SYMBOL_FILTER)

FLOOR_DATE = 2000-01-01
SYMBOL_FILTER = None


In [14]:
# Baseline check for symbols currently used in explore notebook setup
BASELINE_TICKERS = ['CMG', 'SBUX', 'MCD']
baseline_sql = '(' + ', '.join([f"'{t}'" for t in BASELINE_TICKERS]) + ')'

baseline = con.execute(f"""
    SELECT
        symbol,
        COUNT(*) AS rows,
        MIN(trade_date) AS first_date,
        MAX(trade_date) AS last_date
    FROM {silver('fact_daily_prices/**/*.parquet')}
    WHERE symbol IN {baseline_sql}
    GROUP BY symbol
    ORDER BY symbol
""").df()

baseline

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,symbol,rows,first_date,last_date
0,CMG,5137,2006-01-26,2026-06-29
1,MCD,6704,1999-11-01,2026-06-29
2,SBUX,6704,1999-11-01,2026-06-29


In [15]:
# Canonical all-symbol coverage audit
audit_query = f"""
WITH dim AS (
    SELECT
        d.symbol,
        d.ipo_date,
        GREATEST(COALESCE(d.ipo_date, DATE '{FLOOR_DATE}'), DATE '{FLOOR_DATE}') AS expected_start_date
    FROM {silver('dim_company/*.parquet')} d
    {symbol_where}
),
prices AS (
    SELECT
        p.symbol,
        MIN(p.trade_date) AS observed_first_trade_date,
        MAX(p.trade_date) AS observed_last_trade_date,
        COUNT(*) AS observed_rows
    FROM {silver('fact_daily_prices/**/*.parquet')} p
    GROUP BY p.symbol
),
joined AS (
    SELECT
        d.symbol,
        d.ipo_date,
        d.expected_start_date,
        p.observed_first_trade_date,
        p.observed_last_trade_date,
        COALESCE(p.observed_rows, 0) AS observed_rows,
        CASE
            WHEN p.observed_first_trade_date IS NULL THEN NULL
            ELSE datediff('day', d.expected_start_date, p.observed_first_trade_date)
        END AS delta_days
    FROM dim d
    LEFT JOIN prices p ON d.symbol = p.symbol
)
SELECT
    symbol,
    ipo_date,
    expected_start_date,
    observed_first_trade_date,
    observed_last_trade_date,
    observed_rows,
    delta_days,
    (observed_rows = 0) AS missing_prices,
    (delta_days > 0) AS starts_after_expected,
    (delta_days < 0) AS starts_before_expected,
    (observed_rows > 0 AND delta_days = 0) AS coverage_ok
FROM joined
ORDER BY starts_after_expected DESC, delta_days DESC NULLS LAST, symbol
"""

df_audit = con.execute(audit_query).df()
print('Rows in audit:', len(df_audit))
df_audit.head(20)

Rows in audit: 1092


,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,missing_prices,starts_after_expected,starts_before_expected,coverage_ok
0,AA,2016-10-18,2016-10-18,2016-10-18,2026-06-29,2436,0,False,False,False,True
1,AAL,2005-09-27,2005-09-27,2005-09-27,2026-06-29,5220,0,False,False,False,True
2,ABBV,2013-01-02,2013-01-02,2013-01-02,2026-06-29,3392,0,False,False,False,True
3,ABNB,2020-12-10,2020-12-10,2020-12-10,2026-06-29,1392,0,False,False,False,True
4,ACA,2018-10-30,2018-10-30,2018-10-30,2026-06-29,1924,0,False,False,False,True
5,ACI,2020-06-26,2020-06-26,2020-06-26,2026-06-29,1508,0,False,False,False,True
6,ACLS,2000-07-11,2000-07-11,2000-07-11,2026-06-29,6530,0,False,False,False,True
7,ACM,2007-05-10,2007-05-10,2007-05-10,2026-06-29,4814,0,False,False,False,True
8,ACMR,2017-11-03,2017-11-03,2017-11-03,2026-06-29,2172,0,False,False,False,True
9,ACN,2001-07-19,2001-07-19,2001-07-19,2026-06-29,6272,0,False,False,False,True


In [16]:
# Summary metrics
summary = con.execute("""
    SELECT
        COUNT(*) AS total_symbols,
        SUM(CASE WHEN missing_prices THEN 1 ELSE 0 END) AS symbols_missing_prices,
        SUM(CASE WHEN starts_after_expected THEN 1 ELSE 0 END) AS symbols_starting_late,
        ROUND(100.0 * SUM(CASE WHEN starts_after_expected THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_starting_late,
        quantile_cont(delta_days, 0.50) FILTER (WHERE starts_after_expected) AS p50_late_days,
        quantile_cont(delta_days, 0.95) FILTER (WHERE starts_after_expected) AS p95_late_days,
        quantile_cont(delta_days, 0.99) FILTER (WHERE starts_after_expected) AS p99_late_days
    FROM df_audit
""").df()

summary

,total_symbols,symbols_missing_prices,symbols_starting_late,pct_starting_late,p50_late_days,p95_late_days,p99_late_days
0,1092,0.0,0.0,0.0,NaN,NaN,NaN


In [17]:
# Triage tables
top_late = con.execute(f"""
    SELECT *
    FROM df_audit
    WHERE starts_after_expected
    ORDER BY delta_days DESC, symbol
    LIMIT {TOP_N}
""").df()

missing = con.execute(f"""
    SELECT *
    FROM df_audit
    WHERE missing_prices
    ORDER BY symbol
    LIMIT {TOP_N}
""").df()

before_expected = con.execute(f"""
    SELECT *
    FROM df_audit
    WHERE starts_before_expected
    ORDER BY delta_days ASC, symbol
    LIMIT {TOP_N}
""").df()

print('Top late-start symbols:')
display(top_late)
print('Missing prices symbols:')
display(missing)
print('Starts before expected symbols:')
display(before_expected)

Top late-start symbols:


,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,missing_prices,starts_after_expected,starts_before_expected,coverage_ok


Missing prices symbols:


,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,missing_prices,starts_after_expected,starts_before_expected,coverage_ok


Starts before expected symbols:


,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,missing_prices,starts_after_expected,starts_before_expected,coverage_ok
0,ULTI,2025-10-31,2025-10-31,1999-11-01,2026-06-29,5071,-9496,False,False,True,False
1,AAON,1992-12-16,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
2,AAPL,1980-12-12,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
3,ABCB,1994-05-19,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
4,ABT,1983-04-06,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
5,ACGL,1995-09-14,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
6,ACIW,1995-02-27,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
7,ADBE,1986-08-14,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
8,ADC,1994-04-15,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False
9,ADI,1984-07-19,2000-01-01,1999-11-01,2026-06-29,6704,-61,False,False,True,False


In [8]:
# Spot-check sample across categories
sample = con.execute("""
    WITH ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY symbol) AS rn_all,
               ROW_NUMBER() OVER (ORDER BY delta_days DESC NULLS LAST, symbol) AS rn_late
        FROM df_audit
    )
    SELECT * FROM ranked WHERE
        rn_late <= 4
        OR (ipo_date >= DATE '2018-01-01' AND rn_all % 13 = 0)
        OR (ipo_date IS NULL AND rn_all % 7 = 0)
        OR (missing_prices AND rn_all % 5 = 0)
    LIMIT 10
""").df()

sample

,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,missing_prices,starts_after_expected,starts_before_expected,coverage_ok,rn_all,rn_late
0,WDC,1987-01-02,1999-01-01,2026-01-14,2026-06-29,114,9875,False,True,False,False,1048,1
1,AAON,1992-12-16,1999-01-01,1999-11-01,2026-06-29,6704,304,False,True,False,False,4,2
2,AAPL,1980-12-12,1999-01-01,1999-11-01,2026-06-29,6704,304,False,True,False,False,5,3
3,ABCB,1994-05-19,1999-01-01,1999-11-01,2026-06-29,6704,304,False,True,False,False,7,4
4,AX,2018-10-01,2018-10-01,2018-10-01,2026-06-29,1945,0,False,False,False,True,104,649
5,BRW-R,2025-10-08,2025-10-08,2025-10-08,2025-10-27,14,0,False,False,False,True,156,672
6,DOUG,2021-12-30,2021-12-30,2021-12-30,2026-06-29,1127,0,False,False,False,True,299,734
7,EHI-R,2024-09-12,2024-09-12,2024-09-12,2024-10-07,18,0,False,False,False,True,325,745
8,TKO,2023-09-12,2023-09-12,2023-09-12,2026-06-29,701,0,False,False,False,True,949,1025
9,VRAX,2022-07-21,2022-07-21,2022-07-21,2026-06-29,988,0,False,False,False,True,1027,1059


In [ ]:
# Optional: save outputs for tracking
out_dir = PROJECT_ROOT / 'notebooks' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

df_audit.to_csv(out_dir / 'price_coverage_audit.csv', index=False)
summary.to_csv(out_dir / 'price_coverage_summary.csv', index=False)
top_late.to_csv(out_dir / 'price_coverage_top_late.csv', index=False)

print('Wrote CSV outputs to', out_dir)